# GHG-YLL Grid Sensitivity Heatmaps

Heatmap visualizations of the GHG price × YLL value grid sensitivity analysis from the `ghg_yll_grid` config.

Three panels:
- (a) Health burden: Health burden in MYLL
- (b) GHG emissions: GHG emissions in GtCO2eq
- (c) System cost: Total system cost (excl. consumer values and slack) at reference valuations ($10k/YLL, $200/tCO2eq)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sensitivity_utils import (
    FIGURE_WIDTH_MM,
    MM_TO_INCH,
    extract_grid_scenarios,
    load_grid_data_from_statistics,
    pivot_grid_data,
    plot_contour,
    plot_heatmap,
)

In [ ]:
# Configuration
PROJECT_ROOT = Path("..").resolve()
CONFIG_NAME = "ghg_yll_grid"
# CONFIG_NAME = "current_yields"
RESULTS_DIR = PROJECT_ROOT / "results" / CONFIG_NAME

# Constants for recalculating costs
CONSTANT_HEALTH_VALUE_PER_YLL = 10_000  # USD/YLL
CONSTANT_GHG_PRICE = 200  # USD/tCO2eq

## Load Grid Data

In [ ]:
# Extract grid scenarios from config
grid_scenarios_all = extract_grid_scenarios(
    PROJECT_ROOT,
    CONFIG_NAME,
    ghg_param_path=["emissions", "ghg_price"],
    yll_param_path=["health", "value_per_yll"],
    scenario_prefix="ghg",
)

# Filter to only include scenarios with existing network files
grid_scenarios_all = [(g, y, s, f) for g, y, s, f in grid_scenarios_all if f.exists()]

# All extracted grid scenarios are non-baseline (extract_grid_scenarios skips baseline)
grid_scenarios = grid_scenarios_all

# Load baseline from its known solved network file
baseline_path = RESULTS_DIR / "solved" / "model_scen-baseline.nc"
if baseline_path.exists():
    baseline_scenario = [(0.0, 0.0, "baseline", baseline_path)]
    print(f"Baseline scenario: found at {baseline_path}")
else:
    baseline_scenario = []
    print("Baseline scenario: not found")

print(f"Found {len(grid_scenarios)} grid scenarios (excluding baseline)")

# Show unique GHG and YLL values
ghg_values = sorted({g for g, _, _, _ in grid_scenarios})
yll_values = sorted({y for _, y, _, _ in grid_scenarios})
print(f"GHG prices: {ghg_values}")
print(f"YLL values: {yll_values}")

In [ ]:
# Load grid data from pre-computed analysis CSVs
df_grid = load_grid_data_from_statistics(grid_scenarios, PROJECT_ROOT, CONFIG_NAME)
print(f"Grid data shape: {df_grid.shape}")
print(f"Columns: {df_grid.columns.tolist()}")

# Load baseline data
if baseline_scenario:
    df_baseline = load_grid_data_from_statistics(
        baseline_scenario, PROJECT_ROOT, CONFIG_NAME
    )
    print("Baseline data loaded")
else:
    df_baseline = None
    print("No baseline scenario found")

df_grid.head()

## Compute Derived Quantities

In [ ]:
# Compute health cost (billion USD) at constant $10k/YLL
# health_myll is in MYLL, so multiply by 1e6 to get YLL, then by price, then by 1e-9 for billion USD
df_grid["health_cost_bnusd"] = (
    df_grid["health_myll"] * 1e6 * CONSTANT_HEALTH_VALUE_PER_YLL * 1e-9
)

# Compute GHG cost (billion USD) at constant $100/tCO2eq
# ghg_mtco2eq is in MtCO2eq, so multiply by 1e6 to get tCO2eq, then by price, then by 1e-9 for billion USD
df_grid["ghg_cost_bnusd"] = df_grid["ghg_mtco2eq"] * 1e6 * CONSTANT_GHG_PRICE * 1e-9

# Compute system cost using constant values (excluding consumer values)
# Includes: crop production + trade + fertilizer + health cost + GHG cost
df_grid["system_cost"] = (
    df_grid["crop_production"]
    + df_grid["trade"]
    + df_grid["fertilizer"]
    + df_grid["health_cost_bnusd"]
    + df_grid["ghg_cost_bnusd"]
)

# Extract baseline values
if df_baseline is not None:
    baseline_health_myll = df_baseline["health_myll"].iloc[0]
    baseline_ghg_mtco2eq = df_baseline["ghg_mtco2eq"].iloc[0]
    baseline_crop_production = df_baseline["crop_production"].iloc[0]
    baseline_trade = df_baseline["trade"].iloc[0]
    baseline_fertilizer = df_baseline["fertilizer"].iloc[0]

    # Compute baseline system cost (excluding consumer values) at constant externality prices
    baseline_health_cost_bnusd = (
        baseline_health_myll * 1e6 * CONSTANT_HEALTH_VALUE_PER_YLL * 1e-9
    )
    baseline_ghg_cost_bnusd = baseline_ghg_mtco2eq * 1e6 * CONSTANT_GHG_PRICE * 1e-9
    baseline_system_cost = (
        baseline_crop_production
        + baseline_trade
        + baseline_fertilizer
        + baseline_health_cost_bnusd
        + baseline_ghg_cost_bnusd
    )

    # Compute externality savings (absolute, for reference)
    df_grid["externality_savings_bnusd"] = (
        (baseline_health_myll - df_grid["health_myll"])
        * df_grid.index.get_level_values("yll_value")
        + (baseline_ghg_mtco2eq - df_grid["ghg_mtco2eq"])
        * df_grid.index.get_level_values("ghg_price")
    ) / 1000  # Convert MUSD to billion USD

    # Compute system cost ratio: optimized(x,y) / baseline valued at (x,y)
    # For each (ghg_price, yll_value), compute both costs using those valuations
    yll_values = df_grid.index.get_level_values("yll_value")
    ghg_prices = df_grid.index.get_level_values("ghg_price")

    # Optimized system cost at (x,y) valuations (excl. consumer values)
    # health cost = health_myll * yll_value / 1000 (MYLL * USD/YLL / 1000 = billion USD)
    # ghg cost = ghg_mtco2eq * ghg_price / 1000 (MtCO2eq * USD/tCO2eq / 1000 = billion USD)
    df_grid["system_cost_at_xy"] = (
        df_grid["crop_production"]
        + df_grid["trade"]
        + df_grid["fertilizer"]
        + df_grid["health_myll"] * yll_values / 1000
        + df_grid["ghg_mtco2eq"] * ghg_prices / 1000
    )

    # Baseline system cost at (x,y) valuations (excl. consumer values)
    df_grid["baseline_cost_at_xy"] = (
        baseline_crop_production
        + baseline_trade
        + baseline_fertilizer
        + baseline_health_myll * yll_values / 1000
        + baseline_ghg_mtco2eq * ghg_prices / 1000
    )

    # Cost ratio: optimized / baseline (< 1 means optimization saves money)
    df_grid["cost_ratio"] = (
        df_grid["system_cost_at_xy"] / df_grid["baseline_cost_at_xy"]
    )

    print("Baseline (ghg=0, yll=0) values:")
    print(f"  Health burden: {baseline_health_myll:.1f} MYLL")
    print(f"  GHG emissions: {baseline_ghg_mtco2eq:.1f} MtCO2eq")
    print(
        f"  System cost (excl. consumer values, constant prices): {baseline_system_cost:.1f} billion USD"
    )
else:
    baseline_health_myll = baseline_ghg_mtco2eq = baseline_system_cost = None

print("\nGrid derived quantities:")
print(
    f"  System cost range: {df_grid['system_cost'].min():.1f} - {df_grid['system_cost'].max():.1f} billion USD"
)
print(
    f"  Cost ratio range: {df_grid['cost_ratio'].min():.3f} - {df_grid['cost_ratio'].max():.3f}"
)

## Pivot Data for Heatmaps

In [ ]:
# Pivot data for heatmap plotting (log axes require strictly positive valuation levels)
plot_mask = (df_grid.index.get_level_values("ghg_price") > 0) & (
    df_grid.index.get_level_values("yll_value") > 0
)
df_grid_plot = df_grid[plot_mask].copy()

print(
    f"Using {len(df_grid_plot)} scenarios with positive GHG and YLL values for plotting"
)

heatmap_total_obj = pivot_grid_data(df_grid_plot, "total_objective")
heatmap_system_cost = pivot_grid_data(df_grid_plot, "system_cost")
heatmap_health = pivot_grid_data(df_grid_plot, "health_cost_bnusd")
heatmap_ghg = pivot_grid_data(df_grid_plot, "ghg_cost_bnusd")
heatmap_externality_savings = pivot_grid_data(df_grid_plot, "externality_savings_bnusd")
heatmap_cost_ratio = pivot_grid_data(df_grid_plot, "cost_ratio")
heatmap_savings_fraction = 1 - heatmap_cost_ratio

# Also create direct outcome heatmaps
df_grid_plot["ghg_gtco2eq"] = df_grid_plot["ghg_mtco2eq"] / 1000
heatmap_health_myll = pivot_grid_data(df_grid_plot, "health_myll")
heatmap_ghg_gt = pivot_grid_data(df_grid_plot, "ghg_gtco2eq")


# Fill missing grid points for contour interpolation (cubic interpolation requires finite values)
def _prepare_contour_grid(data, name):
    n_missing = int(data.isna().sum().sum())
    if n_missing == 0:
        return data

    filled = data.interpolate(axis=0, limit_direction="both")
    filled = filled.interpolate(axis=1, limit_direction="both")
    filled = filled.ffill(axis=0).bfill(axis=0).ffill(axis=1).bfill(axis=1)

    if filled.isna().any().any():
        raise ValueError(f"Could not fill all missing values in contour grid '{name}'")

    print(f"Filled {n_missing} missing value(s) in contour grid '{name}'")
    return filled


heatmap_health_myll_contour = _prepare_contour_grid(heatmap_health_myll, "health_myll")
heatmap_ghg_gt_contour = _prepare_contour_grid(heatmap_ghg_gt, "ghg_gtco2eq")
heatmap_system_cost_contour = _prepare_contour_grid(heatmap_system_cost, "system_cost")

## Create Heatmap Figure

In [ ]:
# Figure size using constants from sensitivity_utils
fig_width = FIGURE_WIDTH_MM * MM_TO_INCH
fig_height = 90 * MM_TO_INCH  # Height for ~square panels with colorbars below
fig, axes = plt.subplots(1, 3, figsize=(fig_width, fig_height), sharey=True)

# Panel a: Health burden in MYLL
plot_heatmap(
    heatmap_health_myll,
    axes[0],
    title="Health Burden",
    cbar_label="Health burden [MYLL]",
    cmap="Reds",
    panel_label="a",
    cbar_orientation="horizontal",
)

# Panel b: GHG emissions in GtCO2eq (diverging colormap centered at 0)
ghg_gt_min = np.nanmin(heatmap_ghg_gt.values)
ghg_gt_max = np.nanmax(heatmap_ghg_gt.values)
plot_heatmap(
    heatmap_ghg_gt,
    axes[1],
    title="GHG Emissions",
    cbar_label="GHG emissions [GtCO2eq]",
    cmap="RdBu_r",
    panel_label="b",
    vmin=ghg_gt_min,
    vmax=ghg_gt_max,
    vcenter=0,
    cbar_orientation="horizontal",
)
axes[1].set_ylabel("")  # Remove y-label for shared axis

# Panel c: Total system cost at reference valuations (excl. consumer values and slack)
plot_heatmap(
    heatmap_system_cost,
    axes[2],
    title=f"System Cost\n(YLL=${CONSTANT_HEALTH_VALUE_PER_YLL/1000:.0f}k, GHG=${CONSTANT_GHG_PRICE})",
    cbar_label="System cost [billion USD]",
    cmap="Reds",
    panel_label="c",
    cbar_orientation="horizontal",
)
axes[2].set_ylabel("")  # Remove y-label for shared axis

plt.subplots_adjust(wspace=0.12)

# Save figure
output_dir = PROJECT_ROOT / "notebooks" / "figures"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "ghg_yll_grid_heatmaps.pdf"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
print(f"Saved to: {output_path}")

plt.show()

## Contour Plots

Same data as above, but with smoothed contour lines instead of discrete cells.

In [ ]:
# Figure size using constants from sensitivity_utils
fig_width = FIGURE_WIDTH_MM * MM_TO_INCH
fig_height = 90 * MM_TO_INCH  # Height for ~square panels with colorbars below
fig, axes = plt.subplots(1, 3, figsize=(fig_width, fig_height), sharey=True)

# Gaussian smoothing sigma for contour plots
SIGMA = 10

# Panel a: Health burden in MYLL (auto-selected round contour levels)
plot_contour(
    heatmap_health_myll_contour,
    axes[0],
    title="Health Burden",
    cbar_label="Health burden [MYLL]",
    cmap="Reds",
    panel_label="a",
    n_levels=6,
    label_fmt="%.0f",
    cbar_orientation="horizontal",
    sigma=SIGMA,
)

# Panel b: GHG emissions in GtCO2eq (auto-selected round contour levels, zero-centered)
ghg_gt_min = np.nanmin(heatmap_ghg_gt.values)
ghg_gt_max = np.nanmax(heatmap_ghg_gt.values)
plot_contour(
    heatmap_ghg_gt_contour,
    axes[1],
    title="GHG Emissions",
    cbar_label="GHG emissions [GtCO2eq]",
    cmap="RdBu_r",
    panel_label="b",
    vmin=ghg_gt_min,
    vmax=ghg_gt_max,
    vcenter=0,
    n_levels=9,
    label_fmt="%.0f",
    cbar_orientation="horizontal",
    sigma=SIGMA,
)
axes[1].set_ylabel("")  # Remove y-label for shared axis

# Panel c: Total system cost at reference valuations (excl. consumer values and slack)
plot_contour(
    heatmap_system_cost_contour,
    axes[2],
    title=f"System Cost\n(YLL=${CONSTANT_HEALTH_VALUE_PER_YLL/1000:.0f}k, GHG=${CONSTANT_GHG_PRICE})",
    cbar_label="System cost [billion USD]",
    cmap="Reds",
    panel_label="c",
    n_levels=6,
    label_fmt="%.0f",
    cbar_orientation="horizontal",
    sigma=SIGMA,
)
axes[2].set_ylabel("")  # Remove y-label for shared axis

plt.subplots_adjust(wspace=0.12)

# Save figure
output_path = output_dir / "ghg_yll_grid_contours.pdf"
plt.savefig(output_path, dpi=300, bbox_inches="tight")
print(f"Saved to: {output_path}")

plt.show()

## Summary Statistics

In [ ]:
# Print summary statistics
print("Summary Statistics:")
print("=" * 60)

savings_values = heatmap_savings_fraction.values
health_values = heatmap_health_myll.values
ghg_values = heatmap_ghg_gt.values

print()
print("Savings Fraction (optimized vs baseline at same valuations):")
print(
    f"  Grid min: {np.nanmin(savings_values):.3f} ({np.nanmin(savings_values)*100:.1f}%)"
)
print(
    f"  Grid max: {np.nanmax(savings_values):.3f} ({np.nanmax(savings_values)*100:.1f}%)"
)

print()
print("Health Burden:")
print(f"  Grid min: {np.nanmin(health_values):.1f} MYLL")
print(f"  Grid max: {np.nanmax(health_values):.1f} MYLL")
if baseline_health_myll is not None:
    print(f"  Baseline (0,0): {baseline_health_myll:.1f} MYLL")

print()
print("GHG Emissions:")
print(f"  Grid min: {np.nanmin(ghg_values):.2f} GtCO2eq")
print(f"  Grid max: {np.nanmax(ghg_values):.2f} GtCO2eq")
if baseline_ghg_mtco2eq is not None:
    print(f"  Baseline (0,0): {baseline_ghg_mtco2eq / 1000:.2f} GtCO2eq")

# Find point with maximum savings (ignoring missing points)
max_idx = np.unravel_index(np.nanargmax(savings_values), savings_values.shape)
opt_ghg = heatmap_savings_fraction.index[max_idx[0]]
opt_yll = heatmap_savings_fraction.columns[max_idx[1]]
opt_val = savings_values[max_idx]

print()
print("Maximum cost savings:")
print(f"  GHG price: ${opt_ghg:.0f}/tCO2eq")
print(f"  YLL value: ${opt_yll:.0f}/YLL")
print(f"  Savings: {opt_val:.1%}")

## Externality Savings (Reference)

Monetized health + GHG savings vs baseline (0,0) diet, using each scenario's own valuations.

In [ ]:
# Single panel plot for system cost (for reference only)
# System cost at constant externality prices ($10k/YLL, $100/tCO2eq), excluding consumer values

fig, ax = plt.subplots(figsize=(3, 2.5))

plot_heatmap(
    heatmap_system_cost,
    ax,
    title=f"Optimized System Cost\n(YLL=${CONSTANT_HEALTH_VALUE_PER_YLL/1000:.0f}k, GHG=${CONSTANT_GHG_PRICE})",
    cbar_label="System cost [billion USD]",
    cmap="Oranges",
    cbar_orientation="horizontal",
)

plt.tight_layout()
plt.show()

print(
    f"\nSystem cost range: {heatmap_system_cost.values.min():.1f} - {heatmap_system_cost.values.max():.1f} billion USD"
)